Raw processing and looking at REMIND data, which is future predictions of installed capacity.

Import packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

Load REMIND data sets from raw / future data path

In [2]:
# Load all .mif files from the future folder with REMIND data sets and concatenate them into a single dataframe

def load_all_future_files():
    base_path = Path.cwd().parent
    future_folder = base_path / "data" / "raw" / "future"
    
    dfs = []
    
    for file in future_folder.glob("*.mif"):
        df = pd.read_csv(file, sep=";")
        df["source_file"] = file.stem
        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)

future_data = load_all_future_files()

Characterize data set

In [3]:
# Look at head, tail, and random samples from the data set
future_data.head()
future_data.tail()

# Look at shape, column names and index of the data set
future_data.shape
future_data.columns
future_data.index

RangeIndex(start=0, stop=373034, step=1)

Extract data for the needed years and only include wanted region

In [5]:
# Make scenario column with explicit scenaro names
future_data["Scenario"] = future_data["source_file"].str.extract(r"(SSP1|M-SSP2|L-SSP2)")

# New dataframe with only the columns of interest
future_data_2025_2050 = future_data[["Region", "Variable", "Unit", "2025", "2030", "2035", "2040", "2045", "2050", "Scenario"]]

# Only keep rows with the region EUR (European Union) and NEU (Non-European Union in Europe)
future_data_2025_2050 = future_data_2025_2050[future_data_2025_2050["Region"].isin(["EUR", "NEU", "World"])]

# Check data set after filtering
future_data_2025_2050.head()

,Region,Variable,Unit,2025,2030,2035,2040,2045,2050,Scenario
2,EUR,Agricultural Research Intensity,% of Total GDP,0.0195,0.0127,0.0004,0.0080,0.0165,0.0223,L-SSP2
7,NEU,Agricultural Research Intensity,% of Total GDP,0.0149,0.0198,0.0079,0.0058,0.0025,0.0098,L-SSP2
12,World,Agricultural Research Intensity,% of Total GDP,0.0337,0.0414,0.0443,0.0409,0.0383,0.0296,L-SSP2
15,EUR,Biodiversity|BII,unitless,0.7285,0.7290,0.7359,0.7365,0.7371,0.7366,L-SSP2
20,NEU,Biodiversity|BII,unitless,0.7890,0.7893,0.7893,0.7893,0.7897,0.7904,L-SSP2


Only keep electrolyser stock data, variable Cap|Hydrogen|+|Electricity

In [6]:
# Only keep rows with electrolyser data
future_electrolyser_data = future_data_2025_2050[future_data_2025_2050["Variable"] == "SE|Hydrogen"]

# Check data set after filtering
future_electrolyser_data.head()

,Region,Variable,Unit,2025,2030,2035,2040,2045,2050,Scenario
113162,EUR,SE|Hydrogen,EJ/yr,0.091349,0.253137,0.663548,1.102725,1.438790,1.894315,L-SSP2
113167,NEU,SE|Hydrogen,EJ/yr,0.008703,0.016528,0.045755,0.100213,0.175415,0.260160,L-SSP2
113172,World,SE|Hydrogen,EJ/yr,0.654738,1.859174,4.146645,8.374093,14.166459,21.324666,L-SSP2
166611,EUR,SE|Hydrogen,EJ/yr,0.091349,0.236438,0.569802,0.964615,1.331065,1.865968,M-SSP2
166616,NEU,SE|Hydrogen,EJ/yr,0.008703,0.011684,0.043005,0.086513,0.125226,0.162263,M-SSP2


Sum capacities per scenario

In [16]:
# Create dataframe where the rows are years 2025, 2030, 2035, 2040, 2045, 2050 and the columns are yearly values for Region "World"
world_m_ssp2 = future_electrolyser_data[
    (future_electrolyser_data["Region"] == "World") & 
    (future_electrolyser_data["Scenario"] == "M-SSP2")
][["2025", "2030", "2035", "2040", "2045", "2050"]].T

world_m_ssp2.columns = ["Value"]
world_m_ssp2.index.name = "Year"
world_m_ssp2 = world_m_ssp2.reset_index()

print(world_m_ssp2)

   Year      Value
0  2025   0.654738
1  2030   1.811173
2  2035   3.724616
3  2040   6.534135
4  2045   9.453182
5  2050  12.765824
